## Probing: can frozen encodings tell tumor from normal (CAMELYON17)?

CAMELYON17's actual point isn't just tumor/normal classification -- it's *inter-center* generalization: 5 hospitals, different stains/scanners, and the benchmark cares whether a model trained on some centers still works on others. So this notebook fits a linear probe twice:

1. a normal random train/test split (in-distribution, all 5 centers mixed), and
2. **leave-one-center-out**: train on 4 centers, test on the 5th, repeated for each held-out center -- the actual generalization gap a frozen encoder would face in deployment at a new hospital.

In [ ]:
# Install dependencies, then restart the kernel. This is only needed once per environment.
# !pip install -U -r /home/shared/helper/requirements.txt

In [ ]:
import sys

sys.path.append("/home/shared/helper/")
from nbhelper import (
    plt,
    pd,
    np,
)

from probing import probe_metrics, plot_confusion_matrix, latest_matching  # noqa: E402

from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

### 1. Load encodings + metadata

Loads the shared, full CAMELYON17 encodings from `/home/shared/data/camelyon17/encodings/` -- produced by `00_preparation/01_encodings/encode_camelyon.ipynb` -- rather than each participant's own (sampled) run. Row order in the `.npy` matches `metadata_df` row order by construction.

In [ ]:
camelyon_base_dir = Path("/home/shared/data/camelyon17/camelyon17_v1.0/")
encodings_dir = camelyon_base_dir / "encodings"

metadata_path = latest_matching(encodings_dir, "metadata_n*.csv")
metadata_df = pd.read_csv(metadata_path)
suffix = metadata_path.stem.removeprefix("metadata_")

MODEL_NAMES = ["uni2-h", "virchow2"]
features = {}
for model_name in MODEL_NAMES:
    feat_path = encodings_dir / f"{model_name}_{suffix}.npy"
    features[model_name] = np.load(feat_path)
    assert len(features[model_name]) == len(metadata_df)
    print(f"{model_name:10s} <- {feat_path.name}  ({len(metadata_df):,} patches)")

print("\ncenters:", sorted(metadata_df["center"].unique()))
metadata_df["tumor"].value_counts()

In [ ]:
# limit metadata_df and features to 10_000 random samples per center to speed up training and evaluation
metadata_df = metadata_df.groupby("center").sample(n=10_000, random_state=42)
for model_name in MODEL_NAMES:
    features[model_name] = features[model_name][metadata_df.index]

### 2. In-distribution linear probe

Random 80/20 split across all centers, stratified by tumor label. `fit_linear_probe` is defined right here (not imported from `probing.py`) and reused below for the leave-one-center-out split too.

In [ ]:
metadata_df

In [ ]:
def fit_linear_probe(X_train, y_train, X_test, y_test, labels=None, max_iter=2000, seed=42):
    """Standardize + multinomial logistic regression -- the standard frozen-
    feature ("linear probe") evaluation for self-supervised encoders."""
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=max_iter, random_state=seed))
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return clf, probe_metrics(y_test, y_pred, labels=labels)


in_dist_results = []
in_dist_confusions = {}

for model_name in MODEL_NAMES:
    X = features[model_name]
    y = metadata_df["tumor"].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    _, metrics = fit_linear_probe(X_train, y_train, X_test, y_test, labels=[0, 1])
    in_dist_results.append(
        {"encoder": model_name, "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"]}
    )
    in_dist_confusions[model_name] = metrics["confusion_matrix"]

in_dist_df = pd.DataFrame(in_dist_results).set_index("encoder")
in_dist_df

In [ ]:
fig, axs = plt.subplots(1, len(MODEL_NAMES), figsize=(5 * len(MODEL_NAMES), 4.5))
for ax, model_name in zip(axs, MODEL_NAMES):
    plot_confusion_matrix(
        in_dist_confusions[model_name],
        ["normal", "tumor"],
        ax,
        title=f"{model_name} -- in-distribution",
    )
plt.tight_layout()
plt.show()

#### How do misqualified images look like?

### 3. Leave-one-center-out generalization

For each center, train the linear probe on the other 4 and test on the held-out one -- repeated per encoder. This is the honest version of the tumor-detection question: does the probe generalize to a hospital it never saw features from?

In [ ]:
centers = sorted(metadata_df["center"].unique())
looc_results = []

for model_name in MODEL_NAMES:
    X = features[model_name]
    y = metadata_df["tumor"].values
    center = metadata_df["center"].values

    for held_out in centers:
        train_mask = center != held_out
        test_mask = center == held_out
        _, metrics = fit_linear_probe(
            X[train_mask], y[train_mask], X[test_mask], y[test_mask], labels=[0, 1]
        )
        looc_results.append(
            {
                "encoder": model_name,
                "held_out_center": held_out,
                "accuracy": metrics["accuracy"],
                "macro_f1": metrics["macro_f1"],
            }
        )

looc_df = pd.DataFrame(looc_results)
looc_df.pivot(index="held_out_center", columns="encoder", values="accuracy")

### 4. In-distribution vs. cross-center accuracy

The gap between the dashed (in-distribution) line and the per-center bars is the generalization cost of a genuinely unseen hospital.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
width = 0.35
x = np.arange(len(centers))

for i, model_name in enumerate(MODEL_NAMES):
    accs = (
        looc_df[looc_df["encoder"] == model_name]
        .set_index("held_out_center")
        .loc[centers, "accuracy"]
    )
    ax.bar(x + i * width, accs, width, label=f"{model_name} (held-out center)")
    ax.axhline(
        in_dist_df.loc[model_name, "accuracy"],
        color=f"C{i}",
        linestyle="--",
        alpha=0.7,
        label=f"{model_name} (in-distribution)",
    )

ax.set_xticks(x + width / 2)
ax.set_xticklabels([f"center {c}" for c in centers])
ax.set_ylabel("tumor/normal accuracy")
ax.set_ylim(0.5, 1.0)
ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_title("in-distribution vs. leave-one-center-out linear probe")
plt.tight_layout()
plt.show()

#### Qualitative: Show failed images for different centers

### Takeaways

- If cross-center accuracy tracks in-distribution accuracy closely, the encoder has learned tumor morphology that's fairly stain/scanner-invariant.
- A center that drops much further than the others is the interesting case -- worth a manual look at its patches (stain intensity, scanner artifacts) in `01_exploration/explore_camelyon.ipynb`.
- Whichever encoder has the smaller in-distribution/cross-center gap is the more robust one for this task, independent of which has the higher raw accuracy.
- The next notebook asks the flip side of this question: can a probe read *center identity itself* out of the embeddings? If so, that's a shortcut the model could exploit instead of tumor morphology.